In [24]:
import os 
import getpass
import pydantic
from dotenv import load_dotenv
from crewai import LLM, Agent, Task, Crew
from chromadb.config import Settings
from crewai.tools import tool
from com.example.agentic.agent.LLMManager import LLMManager
from com.example.agentic.tools.config import _tool_config, _rag_tool_config

# groq, openai
llm = LLMManager.create_llm('ollama')

llm.call('Hello')

'How can I assist you today?'

#### LOAD Documents

In [1]:
from com.example.agentic.loader.LoadManager import LoadManager
# Website Data Ingestion 
documents = LoadManager.from_url(["https://docs.crewai.com/how-to/Installing-CrewAI/"])

USER_AGENT environment variable not set, consider setting it to identify your requests.


[DEBUG] Found 1 page : ['https://docs.crewai.com/how-to/Installing-CrewAI/']
[DEBUG] Loading pages from : https://docs.crewai.com/how-to/Installing-CrewAI/
[DEBUG] Loaded 1 Web documents from https://docs.crewai.com/how-to/Installing-CrewAI/


[Document(metadata={'source': 'https://docs.crewai.com/how-to/Installing-CrewAI/'}, page_content='Skip to main content\n\nCrewAI home page\n\nlight logo\n\ndark logo\n\nHomeDocumentationAMPAPI ReferenceExamplesChangelog\n\nWebsite\n\nForum\n\nBlog\n\nCrewGPT\n\nWelcome\n\nCrewAI Documentation\n\nWelcome\n\nCrewAI Documentation\n\nBuild collaborative AI agents, crews, and flows — production ready from day one.\n\nShip multi‑agent systems with confidence\n\nDesign agents, orchestrate crews, and automate flows with guardrails, memory, knowledge, and observability baked in.\n\nGet startedView changelogAPI Reference\n\nGet started\n\nIntroduction\n\nOverview of CrewAI concepts, architecture, and what you can build with agents, crews, and flows.\n\nInstallation\n\nInstall via uv, configure API keys, and set up the CLI for local development.\n\nQuickstart\n\nSpin up your first crew in minutes. Learn the core runtime, project layout, and dev loop.\n\nBuild the basics\n\nAgents\n\nCompose agent

#### Chunkings

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

#
print(len(chunks))


3


##### Embaddings

In [ ]:
from com.example.agentic.embedding.EmbeddingManager import EmbeddingManager
from com.example.agentic.vectors.VectorStoreManager import VectorStoreManager

#chunks=[doc.page_content for doc in documents]

### Convert the text to embeddings
embedding_manager=EmbeddingManager()
embeddings = embedding_manager.embed_documents(documents)
print("[INFO] Example embedding:", embeddings[0] if len(embeddings) > 0 else None)

##store int the vector dtaabase
vectorstore = VectorStoreManager()
vectorstore.add_documents(documents,embeddings)

#### Formatters

In [12]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool
from pydantic import BaseModel, Field
from typing import List, Dict
from datetime import datetime

class ResearchPoint(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    findings: str = Field(description="The key findings or insights about this topic")
    relevance: str = Field(description="Why this finding is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for each finding",
        default_factory=list
    )

class ResearchOutput(BaseModel):
    research_points: List[ResearchPoint] = Field(description="List of research findings")
    summary: str = Field(description="Brief summary of overall findings")

class ResearchImage(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    relevance: str = Field(description="Why this image is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="List of images with source_webpage and image_url",
        default_factory=list
    )

class ResearchImageOutput(BaseModel):
    research_images: List[ResearchImage] = Field(description="List of top images on topic")
    summary: str = Field(description="Brief summary of image findings")    

class ExecutiveReportSection(BaseModel):
    section_emoji: str = Field(description="Section emoji (e.g., 🔍, 📊, 🎯)")
    section_title: str = Field(description="Section title")
    section_content: str = Field(description="Main content of the section")
    key_insights: List[str] = Field(description="Key insights from this section")
    recommendations: List[str] = Field(
        default_factory=list,
        description="Optional recommendations based on findings"
    )
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for this section",
        default_factory=list
    )

class ExecutiveReport(BaseModel):
    report_title: str = Field(description="Title of the report")
    generation_date: str = Field(description="Report generation date")
    executive_summary: str = Field(description="A concise executive summary")
    key_findings: List[Dict[str, str]] = Field(
        description="List of key findings with their sources",
        default_factory=list
    )
    report_sections: List[ExecutiveReportSection] = Field(
        description="Detailed report sections"
    )
    next_steps: List[str] = Field(description="Recommended next steps")
    sources: List[Dict[str, str]] = Field(
        description="All sources used in the report",
        default_factory=list
    )

#### Crew Base

In [13]:
from crewai_tools import ScrapeWebsiteTool
from crewai_tools import TavilySearchTool

from crewai.tools import tool
# Perform search for provide topic
websearch_tool = SerperDevTool()

# 1. Initialize the tool with image search enabled
tavily_tool = TavilySearchTool(
    api_key=os.environ.get("TAVILY_API_KEY"),
    include_images=True,
    max_results=10
)
# Initialize the tool to scrape the target website
@tool
def image_scrap_tool():
    """
    scrape images and content for target website
    """
    #website_url='https://www.designdocs.dev/'
    scrape_tool = ScrapeWebsiteTool()

In [5]:
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource

text_kw_source = TextFileKnowledgeSource(
    file_paths=[
        "designs.txt", 
        "machine_learning.txt",
        "python_intro.txt"
    ]
)

##### Design Crew

In [ ]:
from datetime import datetime
from crewai import Task

from crewai_tools import ScrapeWebsiteTool, SerperDevTool
from com.example.agentic.tools.DesignSearchTool import DesignSearchTool, DesignInput
from crewai.knowledge.source.json_knowledge_source import JSONKnowledgeSource

json_source = JSONKnowledgeSource(
    file_paths=["designs-research.json"]
)

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

@CrewBase
class LatestDesignDevelopmentCrew():
    """LatestAiDevelopment crew"""

    agents_config = "config/agents.yaml"
    tasks_config = "config/tasks.yaml"

    # @agent
    # def researcher(self) -> Agent:
    #     return Agent(
    #         config=self.agents_config['researcher'],
    #         verbose=True,
    #         tools=[
    #             SerperDevTool(),
    #             DesignSearchTool()
    #         ],
    #         #knowledge_sources=[text_kw_source],
    #         embedder={
    #             "provider": "openai",
    #             "config": {
    #                 "model": "nomic-embed-text",
    #                 "api_key":"",
    #                 "platform_url":"http://localhost:11434/v1"
    #             },
    #         },
    #         llm=llm
    #     )
    
    @agent
    def image_extractor(self) -> Agent:
        return Agent(
            config=self.agents_config['image_extractor'],
            verbose=True,
            knowledge_sources=[json_source],
            tools=[tavily_tool],
            allow_delegation=False,
            embedder={
                "provider": "openai",
                "config": {
                    "model": "nomic-embed-text",
                    "api_key":"",
                    "platform_url":"http://localhost:11434/v1"
                },
            },
            llm=llm
        )
    
    # @agent
    # def reporting_analyst(self) -> Agent:
    #     return Agent(
    #         config=self.agents_config['reporting_analyst'],
    #         verbose=True,
    #         llm=llm
    #     )
    
    # @agent
    # def formatter(self) -> Agent:
    #     return Agent(
    #         config=self.agents_config['formatter'],
    #         verbose=True,
    #         llm=llm
    #     )

    # @task
    # def research_task(self) -> Task:
    #     return Task(
    #         config=self.tasks_config['research_task'],
    #         output_json=ResearchOutput,
    #         output_file=f'outputs/L005/designs-images_{run_id}.json',
    #     )
    
    @task
    def find_images_task(self) -> Task:
        return Task(
            config=self.tasks_config['find_images_task'],
            output_json=ResearchImageOutput,
            output_file=f'outputs/L005/designs-research_{run_id}.json',
        )
    
    # @task
    # def reporting_task(self) -> Task:
    #     return Task(
    #         config=self.tasks_config['reporting_task'],
    #         output_pydantic=ExecutiveReport,
    #         output_file=f'outputs/L005/designs-report_{run_id}.json',    
    #     )

    # @task
    # def formatting_task(self) -> Task:
    #     return Task(
    #         config=self.tasks_config['formatting_task']
    #         output_file=f'outputs/L005/formatted-design-report_{run_id}.md',
    #     )

		
    @crew
    def crew(self) -> Crew:
        """Creates the LatestDesignDevelopmentCrew crew"""
        return Crew(
            agents=self.agents, # Automatically created by the @agent decorator
            tasks=self.tasks, # Automatically created by the @task decorator
            process=Process.sequential,
            verbose=True,
        )

In [22]:
from datetime import datetime

def run():
    """
    Run the crew.
    """
    inputs = {
        'topic': 'Microservice Design',
        'date': datetime.now().strftime('%Y-%m-%d')
    }
    LatestDesignDevelopmentCrew().crew().kickoff(inputs=inputs)

In [23]:
run()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 5f9f3cb8-6b98-486b-9c69-18ee3b9badce                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: find_images_task                                                                                         │
│  ID: a2b7fe62-0ec4-41c8-a61e-a2f285be1cf2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔍 Knowledge Retrieval ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Knowledge Retrieval Started                                                                                    │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 📚 Knowledge Retrieved ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Search Query:                                                                                                  │
│  {                                                                                                              │
│    "research_images": [                                                                                         │
│      {                                                                                                          │
│        "topic": "Microservice Design",                                                                          │
│        "relevance": "This article discusses the importance of microservices design and how it can improve       │
│  scalability and fault tolerance.",                                                                             │
│        "sources":[                                                                                              │
│          {                                                                                                      │
│            "additionalProperties": false,                                                                       │
│            "type": "object",                                                                                    │
│            "properties": {},                                                                                    │
│            "required": []                                                                                       │
│          },                                                                                                     │
│          {                                                                                                      │
│            "title": "Example Image 1 from Source Webpage",                                                      │
│            "image_url": "https://example.com/image-1.jpg"                                                       │
│          }                                                                                                      │
│        ]                                                                                                        │
│      },                                                                                                         │
│      {                                                                                                          │
│        "topic": "Cloud Computing",                                                                              │
│        "relevance": "This article highlights the benefits of cloud computing for microservice design and        │
│  deployment.",                                                                                                  │
│        "sources":[                                                                                              │
│          {                                                                                                      │
│            "title": "Another Image from Source Webpage",                                                        │
│            "image_url": "https://example.com/image-2.jpg"                                                       │
│          },                                                                                                     │
│          {                                                                                                      │
│            "title": "Example Image 2 from Source Webpage",                                                      │
│            "image_url": "https://example.com/image-2.jpg"                                                       │
│          }                                             

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert Web Image Extractor                                                                              │
│                                                                                                                 │
│  Task: Review the context you got and expands each research_points into full section  make sure you scarp each  │
│  url from research_points's sources and return json list of relavent images.                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert Web Image Extractor                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  research_images=[ResearchImage(topic='DevOps Practices', relevance='The use of containerization and            │
│  automation can improve the overall development speed.', sources=[{'title': 'Huffpost', 'url':                  │
│  'https://www.huffpost.com/national-news/devops-best-practices_N_16921589.htm'}]),                              │
│  ResearchImage(topic='Automation of Microservices Deployment', relevance='Automating deployment involves using  │
│  CI/CD tools to simplify the process.', sources=[{'title': 'Huffpost', 'url':                                   │
│  'https://www.huffpost.com/national-news/how-to-automate-microservices-deployment_N_16921591.htm'}]),           │
│  ResearchImage(topic='Service Discovery', relevance='A service discovery mechanism is required to enable        │
│  communication between services.', sources=[{'title': 'Huffpost', 'url':                                        │
│  'https://www.huffpost.com/national-news/how-to-use-service-discovery_N_16921593.htm'}]),                       │
│  ResearchImage(topic='Monitoring and Error Handling', relevance='Monitoring tools are necessary to track the    │
│  performance of microservices, while error handling mechanisms can prevent errors.', sources=[{'title':         │
│  'Huffpost', 'url': 'https://www.huffpost.com/national-news/how-to-monitor-microservices_N_16921595.htm'}]),    │
│  ResearchImage(topic='Event-Driven Architecture', relevance='An event-driven architecture can use               │
│  publish-subscribe patterns to handle microservice events.', sources=[{'title': 'Huffpost', 'url':              │
│  'https://www.huffpost.com/national-news/how-to-use-event-driven-architecture_N_16921598.htm'}]),               │
│  ResearchImage(topic='Microservices', relevance='The key findings indicate that optimizing microservices        │
│  design is critical for scalability and performance.', sources=[{'title': 'Huffpost', 'url':                    │
│  'https://www.huffpost.com/national-news/microservices-N_16921599.htm'}]), ResearchImage(topic='Scalability',   │
│  relevance='Research highlights the need to optimize microservices infrastructure for scalability and           │
│  performance.', sources=[{'title': 'Huffpost', 'url':                                                           │
│  'https://www.huffpost.com/national-news/scalability-in-microservices_N_16921600.htm'}]),                       │
│  ResearchImage(topic='Performance', relevance='The findings suggest that improving microservice monitoring can  │
│  significantly impact overall system performance.', sources=[{'title': 'Huffpost', 'url':                       │
│  'https://www.huffpost.com/national-news/performance-microservices_N_16921601.htm'}]),                          │
│  ResearchImage(topic='Service Oriented Architecture', relevance='However, the research also emphasizes that     │
│  the integration of services using SOA is crucial for microservices design success.', sources=[{'title':        │
│  'Huffpost', 'url': 'https://www.huffpost.com/national-news/service-oriented-architecture-N_16921602.htm'}]),   │
│  ResearchImage(topic='Microservice Security', relevance='Finally, the research suggests that implementing a     │
│  robust security strategy is essential for protecting microservices against threats.', sources=[{'title':       │
│  'Huffpost', 'url': 'https://www.huffpost.com/national-

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: find_images_task                                                                                         │
│  Agent: Expert Web Image Extractor                                                                              │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 5f9f3cb8-6b98-486b-9c69-18ee3b9badce                                                                       │
│  Final Output: {"research_images":[{"topic":"DevOps Practices","relevance":"The use of containerization and     │
│  automation can improve the overall development                                                                 │
│  speed.","sources":[{"title":"Huffpost","url":"https://www.huffpost.com/national-news/devops-best-practices_N_  │
│  16921589.htm"}]},{"topic":"Automation of Microservices Deployment","relevance":"Automating deployment          │
│  involves using CI/CD tools to simplify the                                                                     │
│  process.","sources":[{"title":"Huffpost","url":"https://www.huffpost.com/national-news/how-to-automate-micros  │
│  ervices-deployment_N_16921591.htm"}]},{"topic":"Service Discovery","relevance":"A service discovery mechanism  │
│  is required to enable communication between                                                                    │
│  services.","sources":[{"title":"Huffpost","url":"https://www.huffpost.com/national-news/how-to-use-service-di  │
│  scovery_N_16921593.htm"}]},{"topic":"Monitoring and Error Handling","relevance":"Monitoring tools are          │
│  necessary to track the performance of microservices, while error handling mechanisms can prevent               │
│  errors.","sources":[{"title":"Huffpost","url":"https://www.huffpost.com/national-news/how-to-monitor-microser  │
│  vices_N_16921595.htm"}]},{"topic":"Event-Driven Architecture","relevance":"An event-driven architecture can    │
│  use publish-subscribe patterns to handle microservice                                                          │
│  events.","sources":[{"title":"Huffpost","url":"https://www.huffpost.com/national-news/how-to-use-event-driven  │
│  -architecture_N_16921598.htm"}]},{"topic":"Microservices","relevance":"The key findings indicate that          │
│  optimizing microservices design is critical for scalability and                                                │
│  performance.","sources":[{"title":"Huffpost","url":"https://www.huffpost.com/national-news/microservices-N_16  │
│  921599.htm"}]},{"topic":"Scalability","relevance":"Research highlights the need to optimize microservices      │
│  infrastructure for scalability and                                                                             │
│  performance.","sources":[{"title":"Huffpost","url":"https://www.huffpost.com/national-news/scalability-in-mic  │
│  roservices_N_16921600.htm"}]},{"topic":"Performance","relevance":"The findings suggest that improving          │
│  microservice monitoring can significantly impact overall system                                                │
│  performance.","sources":[{"title":"Huffpost","url":"https://www.huffpost.com/national-news/performance-micros  │
│  ervices_N_16921601.htm"}]},{"topic":"Service Oriented Architecture","relevance":"However, the research also    │
│  emphasizes that the integration of services using SOA is crucial for microservices design                      │
│  success.","sources":[{"title":"Huffpost","url":"https://www.huffpost.com/national-news/service-oriented-archi  │
│  tecture-N_16921602.htm"}]},{"topic":"Microservice Security","relevance":"Finally, the research suggests that   │
│  implementing a robust security strategy is essential for protecting microservices against                      │
│  threats.","sources":[{"title":"Huffpost","url":"https

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ c05610d9-5f30-461b-9031-fdaaf7653d55                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/c05610d9-5f30-461 │
│ b-9031-fdaaf7653d55?access_code=TRACE-751a0c4eb5                             │
│ 🔑 Access Code: TRACE-751a0c4eb5                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


####
####